In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/bd_grid_version_5.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

divisions = ["Dhaka", "Chattogram", "Rajshahi", "Mymensingh", "Sylhet",
             "Barishal", "Rangpur", "Cumilla", "Khulna"]
for d in divisions:
    df[f"{d}_shed"] = (df[f"{d}_load"] > 0).astype(int)

print(df.shape)   # (1850, 54) - 45 original + 9 target cols


(1850, 54)


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

original_dates = df["Date"].copy()          
full_range = pd.date_range(df["Date"].min(), df["Date"].max())

df_scaffold = df.set_index("Date").reindex(full_range)
df_scaffold.index.name = "Date"
print(df_scaffold.shape)  
# Fill the 17 synthetic gap rows
numeric_cols = df_scaffold.select_dtypes(include=[np.number]).columns
df_scaffold[numeric_cols] = df_scaffold[numeric_cols].interpolate(method="linear", limit_direction="both")

non_numeric_cols = df_scaffold.select_dtypes(exclude=[np.number]).columns
df_scaffold[non_numeric_cols] = df_scaffold[non_numeric_cols].ffill()



(1867, 53)


In [30]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

df_scaffold["day_of_week_num"] = df_scaffold.index.day_of_week
df_scaffold["is_weekend"] = df_scaffold.index.day_of_week.isin([4, 5]).astype(int)
df_scaffold["month"] = df_scaffold.index.month
df_scaffold["is_holiday"] = (df_scaffold["Holiday_cat"] > 0).astype(int)

def season(m):
    if m in [12, 1, 2]: return "winter"
    if m in [3, 4, 5]:  return "summer"
    if m in [6, 7, 8, 9]: return "monsoon"
    return "autumn"
df_scaffold["season"] = df_scaffold["month"].apply(season)
df_scaffold = pd.get_dummies(df_scaffold, columns=["season"], prefix="season")


In [31]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

national_cols = ["Gas/LF limitation", "Coal supply Limitation",
                  "Low water level in Kaptai lake", "Plants under shut down/ maintenance",
                  "Highest Generation (Generation end)", "Minimum Generation (Generation end)",
                  "Max. Demand at eve. peak (Generation end)",
                  "Maximum Temperature in Dhaka was"]

lag_targets = national_cols.copy()
for d in divisions:
    lag_targets += [f"{d}_demand", f"{d}_supply", f"{d}_load"]

lag_features = {}
for col in lag_targets:
    lag_features[f"{col}_lag1"] = df_scaffold[col].shift(1)
    lag_features[f"{col}_lag7"] = df_scaffold[col].shift(7)

df_scaffold = pd.concat([df_scaffold, pd.DataFrame(lag_features, index=df_scaffold.index)], axis=1)
print(df_scaffold.shape)

(1867, 131)


In [32]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

roll_targets = national_cols[:4] + [f"{d}_load" for d in divisions] + [f"{d}_demand" for d in divisions]

roll_features = {}
for col in roll_targets:
    shifted = df_scaffold[col].shift(1)
    roll_features[f"{col}_roll7_mean"]  = shifted.rolling(7,  min_periods=4).mean()
    roll_features[f"{col}_roll7_std"]   = shifted.rolling(7,  min_periods=4).std()
    roll_features[f"{col}_roll14_mean"] = shifted.rolling(14, min_periods=7).mean()

# shedding persistence/momentum signal
for d in divisions:
    shed_shifted = df_scaffold[f"{d}_shed"].shift(1)
    roll_features[f"{d}_shed_count_7d"] = shed_shifted.rolling(7, min_periods=4).sum()

df_scaffold = pd.concat([df_scaffold, pd.DataFrame(roll_features, index=df_scaffold.index)], axis=1)
print(df_scaffold.shape)

(1867, 206)


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

constraint_features = {
    "kaptai_constrained_flag": (df_scaffold["Low water level in Kaptai lake"] > 0).astype(int),
    "plants_down_wow_change": (
        df_scaffold["Plants under shut down/ maintenance"]
        - df_scaffold["Plants under shut down/ maintenance"].shift(7)
    ),
}

df_scaffold = pd.concat([df_scaffold, pd.DataFrame(constraint_features, index=df_scaffold.index)], axis=1)
print(df_scaffold.shape)   

(1867, 208)


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

df_features = df_scaffold.loc[original_dates].reset_index()
print(df_features.shape)   # with all new feature columns


(1850, 209)


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

before = len(df_features)
df_features = df_features.dropna().reset_index(drop=True)
print(f"Dropped {before - len(df_features)} warm-up rows")


Dropped 7 warm-up rows


In [ ]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

leakage_cols = []
for d in divisions:
    leakage_cols += [f"{d}_demand", f"{d}_supply"]   # never used as predictors

df_features = df_features.drop(columns=leakage_cols)


In [37]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

train = df_features[df_features["Date"] <= "2023-12-31"].reset_index(drop=True)
test  = df_features[df_features["Date"] >= "2024-01-01"].reset_index(drop=True)

train.to_csv("../data/processed/train_features.csv", index=False)
test.to_csv("../data/processed/test_features.csv", index=False)
print(train.shape, test.shape)


(1486, 191) (357, 191)


In [38]:
# ==============================================================================
# PROJECT: Predicting Load-Shedding Events in Bangladesh's Power Grid
# AUTHOR: Zahidul Alam (Jeonbuk National University)
# YEAR: 2026
#
# LICENSE: All Rights Reserved.
# This code is proprietary. It may not be copied, distributed, published, or
# used to train machine learning models or generative AI systems without explicit
# written permission from the author.
# ==============================================================================

assert train["Date"].max() < test["Date"].min()

raw_leakage_cols = [f"{d}_demand" for d in divisions] + [f"{d}_supply" for d in divisions]
assert not any(c in df_features.columns for c in raw_leakage_cols), "Raw same-day demand/supply leaked through!"
print("Leakage check passed.")

print("Remaining NaNs:", df_features.isna().sum().sum())
print(df_features.columns.tolist())


Leakage check passed.
Remaining NaNs: 0
['Date', 'Day of the week', 'Year', 'Month', 'Max. Demand at eve. peak (Generation end)', 'Max. Demand at eve. peak (Sub-station end)', 'Highest Generation (Generation end)', 'Minimum Generation (Generation end)', 'Day-peak Generation (Generation end)', 'Evening-peak Generation (Generation end)', 'Minimum Generation Forecast up to 8:00 hrs.', 'Maximum Temperature in Dhaka was', 'Gas/LF limitation', 'Coal supply Limitation', 'Low water level in Kaptai lake', 'Plants under shut down/ maintenance', 'Dhaka_load', 'Chattogram_load', 'Rajshahi_load', 'Mymensingh_load', 'Sylhet_load', 'Barishal_load', 'Rangpur_load', 'Cumilla_load', 'Khulna_load', 'Holiday name', 'Holiday_cat', 'Dhaka_shed', 'Chattogram_shed', 'Rajshahi_shed', 'Mymensingh_shed', 'Sylhet_shed', 'Barishal_shed', 'Rangpur_shed', 'Cumilla_shed', 'Khulna_shed', 'day_of_week_num', 'is_weekend', 'month', 'is_holiday', 'season_autumn', 'season_monsoon', 'season_summer', 'season_winter', 'Gas/LF